# Lesson 12: Attention from scratch to Transformer

Goes with the Lesson 12 page of the Forkcast course. Twin reviews that defeat a bag of words, attention by hand, positional encoding, and a tiny Transformer encoder.

Runtime: CPU is fine; a GPU helps for the Yelp run at the end.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. `forkcast/attention.py`

The twin-review generator and loaders are provided. Write `scaled_dot_product`, `PositionalEncoding` and `TinyTransformer` as the page builds them.

In [ ]:
%%writefile forkcast/attention.py
"""Lesson 12. Attention from scratch to a Transformer encoder.

``make_negation_reviews`` and ``negation_loaders`` are provided; the rest are exercises.
"""

from __future__ import annotations

import math
import random

import torch
import torch.nn.functional as F
from torch import Tensor, nn

from forkcast.text import Vocab, make_text_loaders, tokenize

DISHES = ["pad thai", "pizza", "ramen", "pho", "lasagna", "sushi", "bibimbap", "tacos", "curry", "burger"]
# Each positive template has a negative twin made of exactly the same words in a different order.
PAIRS = [
    ("the {d} was good , not bad .", "the {d} was bad , not good ."),
    ("{d} : not bad , actually good !", "{d} : not good , actually bad !"),
    ("good {d} , not a bad delivery", "bad {d} , not a good delivery"),
    ("great {d} and not terrible service", "terrible {d} and not great service"),
    ("the rider was great , the {d} not terrible", "the rider was terrible , the {d} not great"),
]


def make_negation_reviews(n: int = 600, seed: int = 0) -> tuple[list[str], list[int]]:
    """Reviews where word order decides the label: every positive review has a negative twin with the same words."""
    rng = random.Random(seed)
    texts, labels = [], []
    for i in range(n // 2):
        pos, neg = rng.choice(PAIRS)
        d = rng.choice(DISHES)
        texts += [pos.format(d=d), neg.format(d=d)]
        labels += [1, 0]
    order = list(range(len(texts)))
    rng.shuffle(order)
    return [texts[i] for i in order], [labels[i] for i in order]


def negation_loaders(n_train: int = 600, n_val: int = 200, batch_size: int = 32):
    """Loaders plus the vocabulary, reusing Lesson 11's text pipeline."""
    tr_x, tr_y = make_negation_reviews(n_train, seed=0)
    va_x, va_y = make_negation_reviews(n_val, seed=1)
    vocab = Vocab([tokenize(t) for t in tr_x])
    train_loader, val_loader = make_text_loaders(tr_x, tr_y, va_x, va_y, vocab, batch_size=batch_size)
    return train_loader, val_loader, vocab


def scaled_dot_product(q: Tensor, k: Tensor, v: Tensor, key_mask: Tensor | None = None) -> tuple[Tensor, Tensor]:
    """Attention by hand. q, k, v: (N, T, d). key_mask: (N, T) bool, True on real tokens.

    scores = q @ k^T / sqrt(d)                      (N, T, T): row i says how much token i looks at each token j
    padded keys get -inf before the softmax         (so their weight is exactly 0)
    weights = softmax over the last dimension        each row sums to 1
    output = weights @ v                             (N, T, d)
    Return (output, weights).
    """
    raise NotImplementedError("Lesson 12, step 3: scores, mask with -inf, softmax, weighted sum")


class PositionalEncoding(nn.Module):
    """Adds a fixed sin/cos pattern per position so the model can tell "not good" from "good not".

    Build a table pe of shape (1, max_len, d_model): pe[0, pos, 2i] = sin(pos / 10000^(2i/d)),
    pe[0, pos, 2i+1] = cos(pos / 10000^(2i/d)). Store it with ``self.register_buffer("pe", pe)``:
    part of the model's state, moved with .to(device), saved in the state_dict, but not a parameter.
    forward(x) returns x + pe[:, :x.shape[1]].
    """

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        raise NotImplementedError("Lesson 12, step 4: build pe with torch.arange positions and the 10000 ** (2i/d) divisor; register_buffer")

    def forward(self, x: Tensor) -> Tensor:
        raise NotImplementedError("Lesson 12, step 4: x + self.pe[:, : x.shape[1]]")


class TinyTransformer(nn.Module):
    """Embedding + positions -> a stack of encoder layers -> masked mean -> classifier.

    Use nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True) wrapped in
    nn.TransformerEncoder(layer, num_layers). The encoder's ``src_key_padding_mask`` is True where tokens
    must be IGNORED, so pass ``~mask``. Pool with Lesson 11's masked mean, then a Linear head.
    """

    def __init__(self, vocab_size: int, d_model: int = 32, nhead: int = 2, num_layers: int = 1, dim_feedforward: int = 64, dropout: float = 0.1, n_classes: int = 2):
        super().__init__()
        raise NotImplementedError("Lesson 12, step 5: embedding (padding_idx=0), PositionalEncoding, TransformerEncoder, head")

    def forward(self, ids: Tensor, mask: Tensor) -> Tensor:
        raise NotImplementedError("Lesson 12, step 5: embed, add positions, encoder with src_key_padding_mask=~mask, masked mean, head")

## 3. The bag's ceiling

In [ ]:
import importlib, math, torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from forkcast import attention, text, train
from forkcast.model import count_parameters
importlib.reload(attention)

texts, labels = attention.make_negation_reviews(6, seed=0)
for t, l in zip(texts, labels):
    print(l, t)
train_loader, val_loader, vocab = attention.negation_loaders()
print(len(vocab), "tokens")

def accuracy(model, loader, device="cpu"):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for ids, mask, y in loader:
            correct += (model(ids.to(device), mask.to(device)).argmax(1).cpu() == y).sum().item(); total += len(y)
    return correct / total

torch.manual_seed(0)
bag = text.BagOfEmbeddings(len(vocab), dim=32, hidden=32)
h = train.fit_v2(bag, train_loader, val_loader, epochs=15, lr=3e-3, loss_fn=nn.CrossEntropyLoss())
print("bag: accuracy", accuracy(bag, val_loader), "| validation loss", round(h["val"][-1], 3), "| ln 2 =", round(math.log(2), 3))

## 4. Attention by hand

In [ ]:
importlib.reload(attention)
torch.manual_seed(0)
q, k, v = torch.randn(2, 5, 8), torch.randn(2, 5, 8), torch.randn(2, 5, 8)
out, w = attention.scaled_dot_product(q, k, v)
print(out.shape, w.shape); print(w.sum(dim=-1))
print("matches PyTorch:", torch.allclose(out, F.scaled_dot_product_attention(q, k, v), atol=1e-5))
key_mask = torch.tensor([[True, True, True, False, False]])
_, w = attention.scaled_dot_product(q[:1], k[:1], v[:1], key_mask)
print("weights on the padded keys:", w[0, :, 3:].abs().max().item())

## 5. Positions

In [ ]:
importlib.reload(attention)
pe = attention.PositionalEncoding(32, max_len=50)
print(pe.pe.shape, "| trainable parameters:", sum(p.numel() for p in pe.parameters()), "| in state_dict:", "pe" in pe.state_dict())
plt.figure(figsize=(8, 3)); plt.imshow(pe.pe[0].T, aspect="auto", cmap="RdBu"); plt.xlabel("position"); plt.ylabel("dimension"); plt.colorbar();

## 6. The tiny Transformer

In [ ]:
importlib.reload(attention)
probe = attention.TinyTransformer(vocab_size=50, d_model=16, nhead=2, dropout=0.0).eval()
ids = torch.tensor([[5, 6, 7, 8]]); swapped = torch.tensor([[5, 8, 7, 6]]); padded = torch.tensor([[5, 6, 7, 8, 0, 0]])
print("order changes the output:", not torch.allclose(probe(ids, ids != 0), probe(swapped, swapped != 0)))
print("padding does not:", torch.allclose(probe(ids, ids != 0), probe(padded, padded != 0), atol=1e-5))

torch.manual_seed(0)
model = attention.TinyTransformer(len(vocab), d_model=32, nhead=2, num_layers=1, dim_feedforward=64)
print(count_parameters(model), "parameters")
h = train.fit_v2(model, train_loader, val_loader, epochs=25, lr=3e-3, loss_fn=nn.CrossEntropyLoss())
print("transformer: accuracy", accuracy(model, val_loader), "| validation loss by epoch:", [round(v, 2) for v in h["val"][::5]])

### Look at the attention of one review

In [ ]:
model.eval()
ids, mask, y = next(iter(val_loader))
i = 0; toks = vocab.decode(ids[i][mask[i]]); n = len(toks)
with torch.no_grad():
    x = model.positions(model.embedding(ids[i:i+1, :n]))
    _, w = attention.scaled_dot_product(x, x, x)
plt.figure(figsize=(5, 4.5)); plt.imshow(w[0], cmap="Blues"); plt.xticks(range(n), toks, rotation=90); plt.yticks(range(n), toks); plt.title(f"label {y[i].item()}: {' '.join(toks)}", fontsize=9);

### Break it on purpose: no positions

In [ ]:
class NoPositions(attention.TinyTransformer):
    def __init__(self, *a, **k):
        super().__init__(*a, **k); self.positions = nn.Identity()

torch.manual_seed(0)
blind = NoPositions(len(vocab), d_model=32, nhead=2, num_layers=1, dim_feedforward=64)
train.fit_v2(blind, train_loader, val_loader, epochs=25, lr=3e-3, loss_fn=nn.CrossEntropyLoss())
print("without positional encoding: accuracy", accuracy(blind, val_loader))

## 7. Real reviews: bag against Transformer on Yelp

Reuses Lesson 11's download. Expect a modest gain, not a leap.

In [ ]:
!pip install -q datasets
from datasets import load_dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
stream = load_dataset("fancyzhx/yelp_polarity", split="train", streaming=True).shuffle(seed=0, buffer_size=10_000)
texts, labels = [], []
for ex in stream:
    texts.append(ex["text"]); labels.append(ex["label"])
    if len(texts) >= 40_000:
        break
n_val = 4_000
yelp_vocab = text.Vocab([text.tokenize(t) for t in texts[n_val:]], min_freq=2, max_size=20_000)
y_train, y_val = text.make_text_loaders(texts[n_val:], labels[n_val:], texts[:n_val], labels[:n_val], yelp_vocab, batch_size=128, max_len=200)
torch.manual_seed(0)
bag = text.BagOfEmbeddings(len(yelp_vocab), dim=64, hidden=64)
train.fit_v2(bag, y_train, y_val, epochs=5, lr=3e-3, loss_fn=nn.CrossEntropyLoss(), device=device)
torch.manual_seed(0)
tf = attention.TinyTransformer(len(yelp_vocab), d_model=64, nhead=4, num_layers=2, dim_feedforward=128)
train.fit_v2(tf, y_train, y_val, epochs=5, lr=1e-3, loss_fn=nn.CrossEntropyLoss(), device=device)
print("Yelp validation accuracy: bag", round(accuracy(bag, y_val, device), 3), "| transformer", round(accuracy(tf, y_val, device), 3))

## 8. Definition of done

Expected: `7 passed`, then `90 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_12.py -q

In [ ]:
!pytest -m fast -q

## 9. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 12: attention and a tiny Transformer"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main